# MyHoliday — Destination Dataset Exploration & Merging
**Purpose:** Explore Dataset 1 (560 cities) and Dataset 3 (111 cities), check how many match, then produce a final merged destination table.

---

## Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import json
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print('Libraries loaded successfully.')

## Cell 2 — Load Datasets
> ⚠️ Update the file paths below to match where your CSV files are located.

In [ ]:
# ── Update these paths ──────────────────────────────────────
PATH_DATASET1 = 'dataset1.csv'   # City metadata + thematic ratings
PATH_DATASET2 = 'dataset2.csv'   # Historical trip records
PATH_DATASET3 = 'dataset3.csv'   # Categories + best time to visit
# ────────────────────────────────────────────────────────────

df1 = pd.read_csv(PATH_DATASET1)
df2 = pd.read_csv(PATH_DATASET2)
df3 = pd.read_csv(PATH_DATASET3)

print(f'Dataset 1 shape : {df1.shape}')
print(f'Dataset 2 shape : {df2.shape}')
print(f'Dataset 3 shape : {df3.shape}')

## Cell 3 — Inspect Dataset 1

In [ ]:
print('=== DATASET 1 — Columns ===')
print(df1.columns.tolist())
print()
df1.head(5)

In [ ]:
print('=== DATASET 1 — Data Types ===')
print(df1.dtypes)
print()
print('=== DATASET 1 — Missing Values ===')
print(df1.isnull().sum())

## Cell 4 — Inspect Dataset 2

In [ ]:
print('=== DATASET 2 — Columns ===')
print(df2.columns.tolist())
print()
df2.head(5)

In [ ]:
print('=== DATASET 2 — Data Types ===')
print(df2.dtypes)
print()
print('=== DATASET 2 — Missing Values ===')
print(df2.isnull().sum())

## Cell 5 — Inspect Dataset 3

In [ ]:
print('=== DATASET 3 — Columns ===')
print(df3.columns.tolist())
print()
df3.head(5)

In [ ]:
print('=== DATASET 3 — Data Types ===')
print(df3.dtypes)
print()
print('=== DATASET 3 — Missing Values ===')
print(df3.isnull().sum())

## Cell 6 — Standardise City Names
Before matching, we clean city names: strip whitespace, lowercase, remove special characters.

> ⚠️ Update `city_col_d1` and `city_col_d3` below to match the actual column names in your datasets.

In [ ]:
# ── Update column names to match your actual datasets ───────
city_col_d1 = 'name'    # Column name for city in Dataset 1
city_col_d3 = 'City'    # Column name for city in Dataset 3
# ────────────────────────────────────────────────────────────

def clean_city_name(s):
    if pd.isna(s):
        return ''
    return str(s).strip().lower()

df1['city_clean'] = df1[city_col_d1].apply(clean_city_name)
df3['city_clean'] = df3[city_col_d3].apply(clean_city_name)

print('Sample cleaned names — Dataset 1:')
print(df1['city_clean'].head(10).tolist())
print()
print('Sample cleaned names — Dataset 3:')
print(df3['city_clean'].head(10).tolist())

## Cell 7 — Check How Many Cities Match

In [ ]:
d1_cities = set(df1['city_clean'])
d3_cities = set(df3['city_clean'])

matched     = d1_cities & d3_cities
only_in_d1  = d1_cities - d3_cities
only_in_d3  = d3_cities - d1_cities

print(f'Total cities in Dataset 1   : {len(d1_cities)}')
print(f'Total cities in Dataset 3   : {len(d3_cities)}')
print(f'Matched cities              : {len(matched)}')
print(f'Only in Dataset 1           : {len(only_in_d1)}')
print(f'Only in Dataset 3           : {len(only_in_d3)}')
print()
print('Cities only in Dataset 3 (not in Dataset 1):')
print(sorted(only_in_d3))

In [ ]:
# Visualise match breakdown
labels  = ['Matched\n(both datasets)', 'Only in\nDataset 1', 'Only in\nDataset 3']
values  = [len(matched), len(only_in_d1), len(only_in_d3)]
colors  = ['#c9974f', '#4a7c6b', '#7c4a6b']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='none')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('City Match Analysis — Dataset 1 vs Dataset 3', fontsize=13, fontweight='bold', pad=14)
ax.set_ylabel('Number of Cities')
ax.set_facecolor('#f9f9f7')
fig.patch.set_facecolor('#f9f9f7')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## Cell 8 — Decide Merge Strategy
Based on the match count above:
- **If matched >= 80** → Left Join on Dataset 1 (keep all 560, NULLs for unmatched)
- **If matched < 80** → Inner Join (keep only matched cities for complete data)

This cell will automatically recommend a strategy.

In [ ]:
THRESHOLD = 80

if len(matched) >= THRESHOLD:
    strategy = 'left'
    print(f'✅ Recommendation: LEFT JOIN — Keep all {len(d1_cities)} cities from Dataset 1.')
    print(f'   {len(only_in_d1)} cities will have NULL for categories and best_time_to_visit.')
else:
    strategy = 'inner'
    print(f'⚠️  Recommendation: INNER JOIN — Only keep {len(matched)} matched cities.')
    print(f'   This ensures all cities have complete data.')

print(f'\nSelected strategy: {strategy.upper()} JOIN')

## Cell 9 — Perform the Merge (Dataset 1 + Dataset 3)
> ⚠️ Update `country_col_d3` and `category_col_d3` and `besttime_col_d3` to match your Dataset 3 column names.

In [ ]:
# ── Update column names to match your actual Dataset 3 ──────
category_col_d3   = 'Category'          # e.g. 'history, culture, museums'
besttime_col_d3   = 'Best time to visit' # e.g. 'Apr, May, Jun'
# ────────────────────────────────────────────────────────────

# Select only needed columns from Dataset 3
df3_slim = df3[['city_clean', category_col_d3, besttime_col_d3]].copy()
df3_slim.columns = ['city_clean', 'categories', 'best_time_to_visit']

# Drop duplicates in Dataset 3 (keep first occurrence)
df3_slim = df3_slim.drop_duplicates(subset='city_clean', keep='first')

# Perform merge
df_merged = df1.merge(df3_slim, on='city_clean', how=strategy)

print(f'Merged dataset shape: {df_merged.shape}')
print(f'Null counts after merge:')
print(df_merged[['categories', 'best_time_to_visit']].isnull().sum())
df_merged.head(5)

## Cell 10 — Clean Dataset 2 (Historical Trips)

In [ ]:
# ── Update column names to match your actual Dataset 2 ──────
cols_to_keep = [
    'Destination',
    'Duration (days)',
    'Traveler Age',
    'Traveler Gender',
    'Traveler Nationality',
    'Accommodation Type',
    'Accommodation Cost (USD)',
    'Transportation Type',
    'Transportation Cost (USD)',
]
# ────────────────────────────────────────────────────────────

# Keep only useful columns
# Filter to only columns that exist (in case names differ slightly)
existing_cols = [c for c in cols_to_keep if c in df2.columns]
missing_cols  = [c for c in cols_to_keep if c not in df2.columns]

if missing_cols:
    print(f'⚠️  These columns were not found in Dataset 2 — update the names above:')
    print(missing_cols)

df2_clean = df2[existing_cols].copy()

# Standardise column names to snake_case
df2_clean.columns = (
    df2_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

# Drop rows with all nulls
df2_clean.dropna(how='all', inplace=True)

print(f'Dataset 2 cleaned shape: {df2_clean.shape}')
print(f'Columns: {df2_clean.columns.tolist()}')
df2_clean.head(5)

## Cell 11 — Quick EDA on Merged Destination Table

In [ ]:
# ── Update these to match your actual column names ──────────
budget_col   = 'budget_level'   # 'Budget', 'Mid-range', 'Luxury'
duration_col = 'ideal_duration' # 'Weekend', 'Short trip', 'Long trip'
region_col   = 'region'
# ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#f9f9f7')

palette = ['#c9974f', '#4a7c6b', '#7c4a6b', '#4a5e7c', '#6b8a3a']

for ax, col, title in zip(
    axes,
    [budget_col, duration_col, region_col],
    ['Budget Level Distribution', 'Ideal Duration Distribution', 'Cities by Region']
):
    if col in df_merged.columns:
        counts = df_merged[col].value_counts()
        bars = ax.bar(counts.index, counts.values,
                      color=palette[:len(counts)], edgecolor='none')
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(val), ha='center', va='bottom', fontsize=9)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_facecolor('#f9f9f7')
        ax.spines[['top','right']].set_visible(False)
        ax.tick_params(axis='x', rotation=30)
    else:
        ax.text(0.5, 0.5, f'Column "{col}" not found\nUpdate column name above',
                ha='center', va='center', transform=ax.transAxes, color='gray')
        ax.set_title(title)

plt.suptitle('Merged Destination Dataset — Overview', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Cell 12 — Quick EDA on Historical Trips (Dataset 2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#f9f9f7')

# Age distribution
age_col = [c for c in df2_clean.columns if 'age' in c]
if age_col:
    axes[0].hist(df2_clean[age_col[0]].dropna(), bins=20, color='#c9974f', edgecolor='none')
    axes[0].set_title('Traveler Age Distribution', fontweight='bold')
    axes[0].set_xlabel('Age')
    axes[0].set_ylabel('Count')
    axes[0].set_facecolor('#f9f9f7')
    axes[0].spines[['top','right']].set_visible(False)

# Gender distribution
gender_col = [c for c in df2_clean.columns if 'gender' in c]
if gender_col:
    counts = df2_clean[gender_col[0]].value_counts()
    axes[1].bar(counts.index, counts.values, color=['#4a7c6b','#7c4a6b'], edgecolor='none')
    axes[1].set_title('Traveler Gender Distribution', fontweight='bold')
    axes[1].set_facecolor('#f9f9f7')
    axes[1].spines[['top','right']].set_visible(False)

# Top 10 destinations
dest_col = [c for c in df2_clean.columns if 'destination' in c]
if dest_col:
    top_dest = df2_clean[dest_col[0]].value_counts().head(10)
    axes[2].barh(top_dest.index[::-1], top_dest.values[::-1], color='#4a5e7c', edgecolor='none')
    axes[2].set_title('Top 10 Destinations (Historical)', fontweight='bold')
    axes[2].set_facecolor('#f9f9f7')
    axes[2].spines[['top','right']].set_visible(False)

plt.suptitle('Historical Trips Dataset — Overview', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Cell 13 — Final Cleanup & Rename Columns for Database
Standardise all column names to snake_case, ready for PostgreSQL import.

In [ ]:
def to_snake_case(col):
    return (
        col.strip()
        .lower()
        .replace(' ', '_')
        .replace('-', '_')
        .replace('(', '')
        .replace(')', '')
    )

df_destinations = df_merged.copy()
df_destinations.drop(columns=['city_clean'], inplace=True)
df_destinations.columns = [to_snake_case(c) for c in df_destinations.columns]

df_historical_trips = df2_clean.copy()

print('=== Final Destination Table Columns ===')
print(df_destinations.columns.tolist())
print(f'Shape: {df_destinations.shape}')
print()
print('=== Final Historical Trips Table Columns ===')
print(df_historical_trips.columns.tolist())
print(f'Shape: {df_historical_trips.shape}')

## Cell 14 — Export Final CSVs

In [ ]:
df_destinations.to_csv('destinations_final.csv', index=False)
df_historical_trips.to_csv('historical_trips_final.csv', index=False)

print('✅ Exported successfully:')
print('   destinations_final.csv      — import this into the destinations table in Supabase')
print('   historical_trips_final.csv  — import this into the historical_trips table in Supabase')

---
## Summary

| Output File | Description | Import Into |
|---|---|---|
| `destinations_final.csv` | Merged Dataset 1 + 3 | `destinations` table |
| `historical_trips_final.csv` | Cleaned Dataset 2 | `historical_trips` table |

**Next step:** Use these CSVs to design your ERD and write your CREATE TABLE SQL scripts.